In [1]:
import polars as pl 

In [2]:
df = pl.read_csv('insurance.csv')
df

age,sex,bmi,children,smoker,region,charges
i64,str,f64,i64,str,str,f64
19,"""female""",27.9,0,"""yes""","""southwest""",16884.924
18,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28,"""male""",33.0,3,"""no""","""southeast""",4449.462
33,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32,"""male""",28.88,0,"""no""","""northwest""",3866.8552
…,…,…,…,…,…,…
50,"""male""",30.97,3,"""no""","""northwest""",10600.5483
18,"""female""",31.92,0,"""no""","""northeast""",2205.9808
18,"""female""",36.85,0,"""no""","""southeast""",1629.8335


In [3]:
df.head()

age,sex,bmi,children,smoker,region,charges
i64,str,f64,i64,str,str,f64
19,"""female""",27.9,0,"""yes""","""southwest""",16884.924
18,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28,"""male""",33.0,3,"""no""","""southeast""",4449.462
33,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32,"""male""",28.88,0,"""no""","""northwest""",3866.8552


In [4]:
# Pisahkan fitur (X) dan target (y).
X = df.drop('charges')
y = df['charges']

In [5]:
import polars.selectors as pl_selectors

In [6]:
# Identifikasi kolom numerik dan kategorikal.
X_numeric = X.select(pl_selectors.numeric()).columns
X_categorical = X.select(~pl_selectors.numeric()).columns
X_numeric, X_categorical

(['age', 'bmi', 'children'], ['sex', 'smoker', 'region'])

In [7]:
from sklearn.model_selection import train_test_split

# Bagi data menjadi train set dan test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train_numeric = X_train[X_numeric]
X_train_categorical = X_train[X_categorical]

X_test_numeric = X_test[X_numeric]
X_test_categorical = X_test[X_categorical]

In [8]:
# Buat preprocessing untuk fitur numerik dan kategorikal.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), X_numeric),
        ('cat', OneHotEncoder(), X_categorical)
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)


In [21]:
from lightgbm import LGBMRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np



In [13]:
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor


In [30]:
import numpy as np
import torch

from xrfm import xRFM
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import train_test_split


class XRFMRegressorAdapter(RegressorMixin, BaseEstimator):
    def __init__(
        self,
        model_kwargs=None,
        val_size=0.15,
        random_state=42,
    ):
        self.model_kwargs = model_kwargs
        self.val_size = val_size
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y).reshape(-1)

        X_fit, X_val, y_fit, y_val = train_test_split(
            X,
            y,
            test_size=self.val_size,
            random_state=self.random_state,
        )

        kwargs = dict(self.model_kwargs or {})
        kwargs.setdefault("tuning_metric", "mse")

        self.model_ = xRFM(**kwargs)

        self.model_.fit(
            X_fit, y_fit,
            X_val, y_val,
        )

        return self

    def predict(self, X):
        pred = self.model_.predict(X)

        if isinstance(pred, torch.Tensor):
            pred = pred.detach().cpu().numpy()

        return np.asarray(pred).reshape(-1)

/home/data/kuliah/rka/ML/Asset/Modul-Praktikum-ML-RKA/5. Supervised Learning Regression/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
xrfm_adapter = XRFMRegressorAdapter(
    model_kwargs={
        "tuning_metric": "mse",
        "verbose": False,
        # Tambahkan konfigurasi xRFM lainnya di sini.
    },
    val_size=0.15,
    random_state=42,
)


In [32]:
from desReg.des.DESRegression import DESRegression

estimator = [
    (LGBMRegressor(verbose=-1, random_state=42)),
    (SVR(kernel='rbf')),
    (XGBRegressor()),
    xrfm_adapter,
]

DES = DESRegression(
    regressors_list=estimator,
    n_estimators_bag=5,
    DSEL_perc=0.25,
    XTRAIN_full=False
)

DES.fit(X_train_preprocessed, y_train)



,regressors_list,"[LGBMRegressor...2, verbose=-1), SVR(), ...]"
,n_estimators_bag,5
,DSEL_perc,0.25
,XTRAIN_full,False
,random_state,None
,n_jobs,-1
,k,5
,distance,<function euc...x7ebe24137f60>
,competence_region,'knn'
,competence_level,<function all...x7ebdae0a68e0>
,regressor_selection,<function mea...x7ebe881b7cf0>


In [33]:
from sklearn.metrics import mean_squared_error, r2_score
y_pred_test = DES.predict(X_test_preprocessed)
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred_test))
print("R-squared (R2):", r2_score(y_test, y_pred_test))

Mean Squared Error (MSE): 21837059.341874685
R-squared (R2): 0.8593414636496929
